In [91]:
import pandas as pd
import json
import matplotlib.pyplot as plt
from pprint import pprint
from services.db import MongoDB_one_zero_four


plt.rcParams['font.family'] = ['Microsoft JhengHei']
plt.rcParams['axes.unicode_minus'] = False

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 100)

In [92]:
db = MongoDB_one_zero_four()

condition = {"header.jobName": {"$regex": "python", "$options": "i"}}
projection = {"header": 1, "condition": 1, "_id": 1}
result = db.select_from_bronze(condition, projection)

In [93]:
test = []
for row in result:
    data = row["condition"]["specialty"]
    data.append({"job_id":row["_id"]})
    test.append(data)

In [94]:
print(test[0])

[{'code': '12001002016', 'description': 'Github'}, {'code': '12001003045', 'description': 'Python'}, {'code': '12001003086', 'description': 'PyTorch'}, {'code': '12001002018', 'description': 'Git'}, {'code': '12001001007', 'description': 'Linux'}, {'job_id': '8uzq4'}]


In [95]:
"""
轉換成 df 時是以每個 list 作為一個 column, 所以 df 是以 0, 1, ... list index 作為 col name, 
stack 之後因為 series 只有一行, 所以變成row[row] 的格式? 這種格式經過 df.to_list() 會以[row[row], row[row]] 這樣被拉平? 
會像是 [{}, {}, {}]? 有點神奇, 經過這些操作後撫平了一層巢狀結構
"""
"""
處理這種結構: [[dict, dict], [dict, dict], [dict, dict]]
"""
df = pd.DataFrame(test)
series = df.stack()
df_clean = pd.DataFrame(series.to_list())
print(df_clean)

# print("------------")
# series = pd.Series(test)
# print(series)

            code description job_id
0    12001002016      Github    NaN
1    12001003045      Python    NaN
2    12001003086     PyTorch    NaN
3    12001002018         Git    NaN
4    12001001007       Linux    NaN
..           ...         ...    ...
414  12001005023         AWS    NaN
415  12001006013        HTML    NaN
416  12001006017  JavaScript    NaN
417  12001006032         CSS    NaN
418          NaN         NaN  7wutg

[419 rows x 3 columns]


In [ ]:
# get job_name with job_id 作為映射主表
job_name_include:list[str] = ["python", "後端", "engineer", "資料工程"]
job_name_include_regex = r"|".join(job_name_include)

condition = {
    "header.jobName":{
        "$regex":job_name_include_regex,
        "$options":"i"
}}
projection = {
    "header":1,
    "_id":1,
    "condition":1
}

result = db.select_from_bronze(condition, projection)
df = pd.DataFrame(result[0:5])
test = df.copy()